# Mistral 7B + LoRA — Lightning AI

**Окружение:** Lightning AI Studio, GPU L4 24 GB или T4 16 GB.  
**Метод:** LoRA без квантования (полные веса в fp16/bf16).  
**Подвыборка train:** 10 000 примеров.  
**Эпох:** 2.  
**Ожидаемое время:** ~1 ч на L4, ~1.5–2 ч на T4.

**Настройка:**

1. На lightning.ai создайте новую Studio.  
2. Выберите GPU (L4 даст лучше скорость).  
3. Откройте Terminal и выполните: `pip install jupyterlab` (если не стоит).  
4. Загрузите этот `.ipynb` в Studio (drag-and-drop).  
5. В Settings → Environment Variables добавьте `HF_TOKEN` со значением своего токена.  
6. Откройте блокнот и запускайте ячейки по порядку.

## Установка и импорты

In [ ]:
# Блок 1. Установка и импорты (для Lightning AI Studio)
!pip install -q -U transformers peft accelerate datasets evaluate scikit-learn huggingface_hub

import os, time, gc, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding, set_seed,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, hamming_loss

SEED = 42
set_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f"GPU count: {n}")
    for i in range(n):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"  GPU {i}: {name}, VRAM = {vram:.1f} GB")


## Логин в Hugging Face

In [ ]:
# Блок 2. Логин в Hugging Face
# В Lightning AI Studio есть 2 способа задать HF_TOKEN:
#  (a) через переменную окружения HF_TOKEN в настройках Studio (рекомендуется)
#  (b) ввести вручную ниже
from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Введите HF_TOKEN: ").strip()

login(token=HF_TOKEN)
print("✅ HF auth OK")


## Параметры эксперимента

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
RUN_NAME   = "mistral7b_lora_lightning"


## Данные с подвыборкой

In [ ]:
# Блок 3. Загрузка GoEmotions + подвыборка для скорости
dataset = load_dataset("go_emotions", "simplified")
print("Исходный размер:", dataset)

SUBSAMPLE_TRAIN = 21705
dataset["train"] = dataset["train"].shuffle(seed=SEED).select(range(SUBSAMPLE_TRAIN))
print(f"\nПосле подвыборки train: {len(dataset['train'])}")

label_names = dataset['train'].features['labels'].feature.names
num_labels = len(label_names)
print(f"Классов: {num_labels}")

def multi_hot(batch):
    arr = np.zeros((len(batch['text']), num_labels), dtype=np.float32)
    for i, lbls in enumerate(batch['labels']):
        arr[i, lbls] = 1.0
    return {"encoded_labels": arr.tolist()}

encoded = dataset.map(multi_hot, batched=True, remove_columns=['labels', 'id'])


## Токенизация

In [ ]:
# Блок 4. Токенизация
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

MAX_LEN = 128

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized = encoded.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized = tokenized.rename_column("encoded_labels", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print(tokenized)


## Загрузка модели

In [ ]:
# Блок 5. Загрузка полной модели в fp16/bf16
# Lightning AI часто даёт L4 (bf16) или T4 (только fp16). Выбираем по факту.
GPU_NAME = torch.cuda.get_device_name(0).lower()
SUPPORTS_BF16 = any(x in GPU_NAME for x in ["a100", "l4", "h100", "rtx 30", "rtx 40", "a10"])
MODEL_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
print(f"GPU: {GPU_NAME} → dtype = {MODEL_DTYPE}")

print(f"Загрузка {MODEL_NAME} ...")
t0 = time.time()
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    torch_dtype=MODEL_DTYPE,
    device_map="auto",
    token=HF_TOKEN,
)
print(f"Модель загружена за {time.time()-t0:.1f} сек")

model.config.pad_token_id = tokenizer.pad_token_id
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    mem = torch.cuda.memory_allocated(i) / 1024**3
    print(f"VRAM GPU {i}: {mem:.2f} GB")


## LoRA

In [ ]:
# Блок 6. LoRA адаптеры
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["score"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## Метрики

In [ ]:
# Блок 7. Метрики
THRESHOLD = 0.5

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= THRESHOLD).astype(int)
    labels = labels.astype(int)
    metrics = {
        "f1_macro":        f1_score(labels, preds, average="macro", zero_division=0),
        "f1_micro":        f1_score(labels, preds, average="micro", zero_division=0),
        "f1_weighted":     f1_score(labels, preds, average="weighted", zero_division=0),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro":    recall_score(labels, preds, average="macro", zero_division=0),
        "hamming_loss":    hamming_loss(labels, preds),
    }
    try:
        metrics["roc_auc_macro"] = roc_auc_score(labels, probs, average="macro")
    except ValueError:
        metrics["roc_auc_macro"] = float("nan")
    return metrics


## Обучение

In [ ]:
# Блок 8. Обучение
OUTPUT_DIR = f"./outputs/{RUN_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,           # эффективный batch = 16
    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="adamw_torch",
    bf16=SUPPORTS_BF16,
    fp16=not SUPPORTS_BF16,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)
t0 = time.time()
train_result = trainer.train()
train_time = time.time() - t0
peak_mem = sum(torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())) / 1024**3

print(f"\n⏱  Время обучения: {train_time/60:.2f} мин")
print(f"📈 Пиковая VRAM (sum):  {peak_mem:.2f} GB")


## Оценка на тесте

In [ ]:
# Блок 9. Финальная оценка на тесте
print("Оценка на тесте ...")
test_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
print("\n=== Тестовые метрики ===")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k:30s}: {v:.4f}")
    else:
        print(f"{k:30s}: {v}")


## Сохранение

In [ ]:
# Блок 10. Сохранение адаптера и сводки
adapter_dir = f"{OUTPUT_DIR}/final_adapter"
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

def folder_size_mb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1024**2

adapter_size = folder_size_mb(adapter_dir)
print(f"💾 Размер адаптера: {adapter_size:.2f} MB")

summary = {
    "run_name": RUN_NAME,
    "model": MODEL_NAME,
    "method": "LoRA (fp16/bf16, без квантования)",
    "lora_r": 16,
    "lora_alpha": 32,
    "train_subsample": SUBSAMPLE_TRAIN,
    "epochs": training_args.num_train_epochs,
    "effective_batch": training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
    "learning_rate": training_args.learning_rate,
    "train_time_min": round(train_time/60, 2),
    "peak_vram_gb_sum_both_gpus": round(peak_mem, 2),
    "adapter_size_mb": round(adapter_size, 2),
    "test_metrics": {k: float(v) for k, v in test_metrics.items() if isinstance(v, (int, float))},
}
with open(f"{OUTPUT_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\n=== ИТОГ ===")
print(json.dumps(summary, indent=2, ensure_ascii=False))
print(f"\n📁 Результаты сохранены в: {OUTPUT_DIR}")
